# RAG with Query Rewriting

## Rewrite Vague Questions Before Retrieval -- Why It Helps and How Much

**Project 22** -- Advance RAG Learning Series

| Property | Value |
|----------|-------|
| Task | Query rewriting for improved RAG retrieval |
| Strategies | Clarification, Step-back, Multi-query expansion |
| Retrieval | Dense embedding search (sentence-transformers) |
| Corpus | 25-document tech knowledge base |
| Evaluation | Recall@3, MRR, before/after comparison |


## Project Overview

Users ask bad questions. They use vague pronouns, conversational shortcuts,
and overloaded multi-part queries. A retrieval system that takes these
queries literally will return irrelevant context -- and the generator
will produce a bad answer.

**Query rewriting** transforms the user's raw question into one or more
better-formed search queries **before** retrieval.

Three strategies:

| Strategy | What it does | When to use |
|----------|-------------|-------------|
| **Clarification** | Make vague queries specific by expanding pronouns and implicit references | Ambiguous questions |
| **Step-back** | Broaden a too-narrow query to retrieve more context | Overly specific queries |
| **Multi-query** | Generate multiple search queries from one complex question | Multi-intent questions |

This notebook benchmarks all three against **raw (no rewriting)** retrieval.


## Learning Goals

1. Understand **why** raw user queries often fail at retrieval
2. Implement three query rewriting strategies (clarification, step-back, multi-query)
3. Measure retrieval quality before and after rewriting
4. Learn when each strategy helps and when it hurts
5. Design rewriting rules that work without an external LLM


## Problem Statement

Given a knowledge base about a software platform, users ask questions like:

- *"How does it work?"* -- what is "it"?
- *"What about pricing?"* -- pricing of what feature?
- *"Compare the databases and also tell me about caching"* -- two separate intents

These vague queries retrieve wrong documents. The goal: **rewrite them
into precise search queries** and measure the improvement.


## Why Query Rewriting Matters

- **Retrieval is the bottleneck of RAG.** If you retrieve wrong docs,
  the generator cannot produce a good answer -- no matter how good the LLM.
- **Users don't write search queries.** They write conversational questions
  with implicit context, pronouns, and multi-part intents.
- **Rewriting is cheap.** One LLM call to rewrite costs far less than
  a bad answer that needs human correction.
- **Composable with other techniques.** Rewriting stacks with hybrid search,
  reranking, and context compression.


## Environment Setup

In [ ]:
import subprocess, sys, warnings

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["sentence-transformers"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        _install(pkg)

warnings.filterwarnings("ignore")
print("Environment ready.")


## Imports

In [ ]:
import re, random, time
from typing import List, Dict, Tuple
from dataclasses import dataclass, field

import numpy as np
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("All imports loaded.")


## Configuration

In [ ]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
K = 3   # retrieve top-K

print(f"Config: model={EMBEDDING_MODEL}, K={K}")


## Knowledge Base

A 25-document corpus about a fictional SaaS platform called **CloudSync**.
Covers pricing, features, integrations, security, APIs, and support.


In [ ]:
corpus = [
    {"id": "D01", "text": "CloudSync is a cloud-based data synchronization platform. It syncs files across devices in real time using a peer-to-peer mesh architecture with end-to-end encryption."},
    {"id": "D02", "text": "CloudSync pricing starts at $9/month for the Starter plan (5 users, 100 GB). The Pro plan is $29/month (25 users, 1 TB). Enterprise pricing is custom and includes dedicated support."},
    {"id": "D03", "text": "CloudSync supports PostgreSQL, MySQL, MongoDB, and Redis as backend data stores. PostgreSQL is the default. MongoDB is available on Pro and Enterprise plans."},
    {"id": "D04", "text": "CloudSync handles failures through automatic retry with exponential backoff. Failed sync operations are queued and retried up to 5 times. Users receive email alerts after 3 consecutive failures."},
    {"id": "D05", "text": "CloudSync integrates with Slack, Microsoft Teams, Jira, and GitHub. The Slack integration sends notifications on sync events. The GitHub integration triggers syncs on push events."},
    {"id": "D06", "text": "CloudSync uses AES-256 encryption for data at rest and TLS 1.3 for data in transit. All encryption keys are managed through AWS KMS. SOC 2 Type II certification is available on Enterprise."},
    {"id": "D07", "text": "The CloudSync REST API supports CRUD operations on sync jobs, user management, and webhook configuration. Authentication uses OAuth 2.0 with JWT tokens. Rate limit is 1000 requests per minute."},
    {"id": "D08", "text": "CloudSync provides a Python SDK and a JavaScript SDK. The Python SDK supports async operations via asyncio. Both SDKs are available on PyPI and npm respectively."},
    {"id": "D09", "text": "CloudSync caching uses a two-tier architecture: L1 cache in Redis for hot data (TTL 5 minutes) and L2 cache in local SSD for warm data (TTL 1 hour). Cache hit ratio averages 94%."},
    {"id": "D10", "text": "CloudSync supports file versioning with up to 30 days of history on Pro and unlimited on Enterprise. Users can restore any previous version through the web dashboard or API."},
    {"id": "D11", "text": "CloudSync monitoring uses Prometheus for metrics collection and Grafana for dashboards. Key metrics include sync latency, throughput, error rate, and queue depth."},
    {"id": "D12", "text": "CloudSync deployment supports Docker containers and Kubernetes. The recommended production setup uses 3 replicas with horizontal pod autoscaling. Helm charts are provided."},
    {"id": "D13", "text": "CloudSync access control uses role-based access control (RBAC) with four roles: Admin, Editor, Viewer, and Auditor. Custom roles are available on Enterprise plans."},
    {"id": "D14", "text": "CloudSync performance benchmarks show 99.9% uptime SLA on Enterprise. Average sync latency is 120ms for files under 10 MB. Large file sync (1 GB+) uses chunked transfer with resumable uploads."},
    {"id": "D15", "text": "CloudSync audit logging records all user actions, API calls, and configuration changes. Logs are retained for 90 days on Pro and 1 year on Enterprise. Export to SIEM systems is supported."},
    {"id": "D16", "text": "CloudSync supports single sign-on (SSO) via SAML 2.0 and OpenID Connect. SSO is available on Pro and Enterprise plans. Multi-factor authentication (MFA) is available on all plans."},
    {"id": "D17", "text": "CloudSync webhook system delivers real-time event notifications via HTTP POST. Supported events include sync_complete, sync_failed, user_added, and file_conflict. Retry logic uses exponential backoff."},
    {"id": "D18", "text": "CloudSync conflict resolution uses a last-writer-wins strategy by default. On Enterprise, users can configure merge policies per folder: last-writer-wins, manual review, or automatic merge for text files."},
    {"id": "D19", "text": "CloudSync data residency options include US, EU, and Asia-Pacific regions. Enterprise customers can specify data must remain within a single region for GDPR compliance."},
    {"id": "D20", "text": "CloudSync backup and disaster recovery uses cross-region replication with RPO of 1 hour and RTO of 4 hours. Automatic failover is available on Enterprise with RPO of 5 minutes."},
    {"id": "D21", "text": "CloudSync supports batch sync jobs for migrating large datasets. Batch jobs run during off-peak hours and support throttling to avoid impact on production traffic."},
    {"id": "D22", "text": "CloudSync mobile apps are available for iOS and Android. Mobile sync supports selective folder sync, offline access, and background upload. Push notifications alert users to sync conflicts."},
    {"id": "D23", "text": "CloudSync admin dashboard provides real-time visibility into user activity, storage usage, sync health, and billing. Admins can manage users, configure policies, and view audit logs."},
    {"id": "D24", "text": "CloudSync support includes community forums on the free plan, email support with 24-hour SLA on Pro, and dedicated account manager with 1-hour SLA on Enterprise."},
    {"id": "D25", "text": "CloudSync compliance certifications include SOC 2 Type II, ISO 27001, HIPAA BAA (Enterprise only), and GDPR. Annual penetration testing is conducted by a third-party firm."},
]

print(f"Knowledge base: {len(corpus)} documents")
for doc in corpus[:3]:
    print(f"  [{doc['id']}] {doc['text'][:60]}...")
print("  ...")


## Dense Retriever

We use sentence-transformers to encode corpus and queries.
Retrieval is cosine similarity over normalised embeddings.


In [ ]:
print(f"Loading embedding model: {EMBEDDING_MODEL}...")
encoder = SentenceTransformer(EMBEDDING_MODEL)

doc_texts = [doc["text"] for doc in corpus]
doc_embeddings = encoder.encode(doc_texts, convert_to_numpy=True, show_progress_bar=False)
doc_embeddings = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)


def retrieve(query: str, k: int = K) -> List[Dict]:
    """Retrieve top-k documents by embedding similarity."""
    q_emb = encoder.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    scores = (doc_embeddings @ q_emb.T).flatten()
    top_idx = np.argsort(scores)[::-1][:k]
    return [{"id": corpus[i]["id"], "score": float(scores[i]), "rank": r + 1}
            for r, i in enumerate(top_idx)]


def retrieve_multi(queries: List[str], k: int = K) -> List[Dict]:
    """Retrieve using multiple queries, merge by best score per doc."""
    all_scores = {}
    for q in queries:
        results = retrieve(q, k=k)
        for r in results:
            if r["id"] not in all_scores or r["score"] > all_scores[r["id"]]:
                all_scores[r["id"]] = r["score"]
    sorted_docs = sorted(all_scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return [{"id": doc_id, "score": score, "rank": i + 1}
            for i, (doc_id, score) in enumerate(sorted_docs)]


# Sanity check
test = retrieve("CloudSync pricing plans", k=3)
print(f"Dense retriever ready (index shape: {doc_embeddings.shape})")
print(f"Test: {[r['id'] for r in test]}")


## Why Raw Queries Fail

Consider: *"How does it handle failures?"*

Problems:
- **"it"** -- what does "it" refer to? The retriever doesn't know.
- **"failures"** -- failure of what? Sync? API? Auth? Network?
- **"handle"** -- retry? alert? log? ignore?

The embedding model does its best with semantic similarity, but
the query is so vague that many documents could plausibly match.

**Rewriting** resolves these ambiguities before retrieval.


## Strategy 1: Clarification Rewriting

**Goal:** Make vague queries specific by:
- Expanding pronouns ("it" -> "CloudSync")
- Adding domain context
- Resolving implicit references

In production, an LLM rewrites the query. Here we use rule-based
patterns to demonstrate the concept transparently.


In [ ]:
def rewrite_clarify(query: str, context: str = "CloudSync platform") -> str:
    """Clarification rewriting: expand pronouns, add domain context."""
    rewritten = query
    
    # Expand pronouns
    pronoun_map = {
        r"\bit\b": context,
        r"\bthis\b": context,
        r"\bthe system\b": context,
        r"\bthe tool\b": context,
        r"\bthe product\b": context,
        r"\bthe service\b": context,
    }
    for pattern, replacement in pronoun_map.items():
        rewritten = re.sub(pattern, replacement, rewritten, flags=re.IGNORECASE)
    
    # Add domain qualifier if query is too short
    words = rewritten.split()
    if len(words) < 5 and context.lower() not in rewritten.lower():
        rewritten = f"{context}: {rewritten}"
    
    # Expand vague terms
    vague_expansions = {
        "pricing": "pricing plans costs monthly subscription",
        "security": "security encryption authentication access control",
        "failures": "failure handling retry error recovery alerts",
        "databases": "database storage PostgreSQL MySQL MongoDB Redis",
    }
    for term, expansion in vague_expansions.items():
        if term in rewritten.lower() and expansion not in rewritten.lower():
            rewritten = f"{rewritten} ({expansion})"
    
    return rewritten


# Demonstrate
examples = [
    "How does it handle failures?",
    "What about pricing?",
    "Tell me about security",
    "Does it support SSO?",
]

print("Strategy 1: Clarification Rewriting")
print("=" * 60)
for q in examples:
    rewritten = rewrite_clarify(q)
    print(f"  Original:  {q}")
    print(f"  Rewritten: {rewritten}")
    print()


## Strategy 2: Step-Back Prompting

**Goal:** When a query is too narrow, broaden it to retrieve
more context. Instead of asking about one specific thing,
ask about the general category.

Example:
- Narrow: "What is the Redis TTL for L1 cache?"
- Step-back: "How does CloudSync caching architecture work?"

The broader query retrieves the document that contains the specific answer.


In [ ]:
def rewrite_stepback(query: str) -> str:
    """Step-back rewriting: broaden narrow queries to general topics."""
    # Map specific terms to broader categories
    stepback_rules = [
        (r"redis|memcache|cache\s*hit|TTL|L1 cache|L2 cache",
         "How does CloudSync caching architecture work?"),
        (r"OAuth|JWT|token|bearer|auth\S*cation",
         "How does CloudSync API authentication and authorization work?"),
        (r"AES|TLS|encryption key|KMS",
         "What encryption and security does CloudSync use?"),
        (r"Helm|pod|replica|autoscal",
         "How is CloudSync deployed in production?"),
        (r"Prometheus|Grafana|metric|dashboard",
         "How does CloudSync monitoring and observability work?"),
        (r"SAML|OpenID|SSO|MFA|multi.factor",
         "What authentication methods does CloudSync support?"),
        (r"SOC|ISO|HIPAA|GDPR|compliance|penetration",
         "What compliance certifications does CloudSync have?"),
        (r"RPO|RTO|failover|disaster|backup",
         "How does CloudSync handle backup and disaster recovery?"),
        (r"conflict|merge|last.writer",
         "How does CloudSync resolve file conflicts during sync?"),
        (r"webhook|event|notification",
         "How does CloudSync send real-time notifications?"),
    ]
    
    for pattern, broad_query in stepback_rules:
        if re.search(pattern, query, re.IGNORECASE):
            return broad_query
    
    # Default: prepend "How does CloudSync" if no rule matches
    return f"How does CloudSync handle: {query}"


# Demonstrate
examples = [
    "What is the Redis TTL for L1 cache?",
    "Does CloudSync support OAuth 2.0 with JWT?",
    "What is the RPO for cross-region replication?",
    "How does last-writer-wins conflict resolution work?",
]

print("Strategy 2: Step-Back Rewriting")
print("=" * 60)
for q in examples:
    rewritten = rewrite_stepback(q)
    print(f"  Narrow:    {q}")
    print(f"  Step-back: {rewritten}")
    print()


## Strategy 3: Multi-Query Expansion

**Goal:** Split complex multi-intent queries into separate
search queries, retrieve for each, then merge results.

Example:
- Complex: "Compare the databases and also tell me about caching"
- Queries: ["CloudSync database support", "CloudSync caching architecture"]

Each sub-query retrieves its own relevant docs. Merging by best score
gives comprehensive coverage.


In [ ]:
def rewrite_multiquery(query: str) -> List[str]:
    """Multi-query expansion: split complex queries into sub-queries."""
    multi_queries = []
    
    # Split on conjunctions and separators
    parts = re.split(r"\band\b|\balso\b|\bplus\b|\badditionally\b|;|,\s*(?=\w+\s+\w+)", 
                     query, flags=re.IGNORECASE)
    parts = [p.strip() for p in parts if len(p.strip()) > 10]
    
    if len(parts) > 1:
        for part in parts:
            # Clarify each part
            clarified = rewrite_clarify(part)
            multi_queries.append(clarified)
    else:
        # Single intent -- generate a synonym variant
        multi_queries.append(query)
        # Add a paraphrase
        rephrase_map = {
            "pricing": "cost subscription plan",
            "security": "encryption authentication protection",
            "database": "data store backend storage",
            "deploy": "installation setup configuration",
            "monitor": "observability metrics dashboards",
            "support": "help desk customer service SLA",
        }
        for keyword, synonyms in rephrase_map.items():
            if keyword in query.lower():
                multi_queries.append(f"CloudSync {synonyms}")
                break
    
    return multi_queries[:3]  # cap at 3 sub-queries


# Demonstrate
examples = [
    "Compare the databases and also tell me about caching",
    "What databases are supported and how much does it cost?",
    "Tell me about security",
]

print("Strategy 3: Multi-Query Expansion")
print("=" * 60)
for q in examples:
    sub_queries = rewrite_multiquery(q)
    print(f"  Complex:     {q}")
    for i, sq in enumerate(sub_queries):
        print(f"  Sub-query {i+1}: {sq}")
    print()


## Evaluation Queries

We define queries with ground-truth relevant documents.
Each query has a **type** indicating why it's hard for raw retrieval.


In [ ]:
eval_queries = [
    # Vague / pronoun queries -- need clarification
    {"query": "How does it handle failures?",
     "relevant": ["D04"], "type": "vague",
     "note": "it = CloudSync, failures = sync failures"},
    {"query": "What about the pricing?",
     "relevant": ["D02"], "type": "vague",
     "note": "pricing of what? CloudSync plans"},
    {"query": "Tell me about security",
     "relevant": ["D06", "D16", "D25"], "type": "vague",
     "note": "security is broad: encryption, SSO, compliance"},
    {"query": "Does the system scale?",
     "relevant": ["D12", "D14"], "type": "vague",
     "note": "system = CloudSync, scale = deployment + performance"},

    # Narrow / specific queries -- need step-back
    {"query": "What is the Redis TTL for the L1 cache?",
     "relevant": ["D09"], "type": "narrow",
     "note": "very specific, but full caching doc has the answer"},
    {"query": "Does CloudSync support SAML 2.0 SSO?",
     "relevant": ["D16"], "type": "narrow",
     "note": "specific protocol, auth doc covers it"},
    {"query": "What is the RPO for disaster recovery?",
     "relevant": ["D20"], "type": "narrow",
     "note": "specific metric, DR doc has full details"},

    # Multi-intent queries -- need decomposition
    {"query": "What databases are supported and how much does it cost?",
     "relevant": ["D03", "D02"], "type": "multi",
     "note": "two separate topics: databases + pricing"},
    {"query": "Tell me about caching and monitoring",
     "relevant": ["D09", "D11"], "type": "multi",
     "note": "two separate topics: caching + monitoring"},
    {"query": "Compare deployment options and also the SDK support",
     "relevant": ["D12", "D08"], "type": "multi",
     "note": "two separate topics: deployment + SDKs"},

    # Well-formed queries (control group -- rewriting should not hurt)
    {"query": "How does CloudSync handle file conflict resolution?",
     "relevant": ["D18"], "type": "clear",
     "note": "already clear, rewriting should be neutral"},
    {"query": "What compliance certifications does CloudSync have?",
     "relevant": ["D25"], "type": "clear",
     "note": "already clear, rewriting should be neutral"},
]

print(f"Evaluation set: {len(eval_queries)} queries")
type_counts = {}
for q in eval_queries:
    type_counts[q["type"]] = type_counts.get(q["type"], 0) + 1
for qtype, count in type_counts.items():
    print(f"  {qtype}: {count} queries")


## Evaluation Metrics

Same as project 21 -- Precision@k, Recall@k, MRR.


In [ ]:
def recall_at_k(retrieved_ids: List[str], relevant_ids: List[str], k: int) -> float:
    top_k = retrieved_ids[:k]
    hits = sum(1 for doc_id in relevant_ids if doc_id in top_k)
    return hits / len(relevant_ids) if relevant_ids else 0.0


def mrr_score(retrieved_ids: List[str], relevant_ids: List[str]) -> float:
    for i, doc_id in enumerate(retrieved_ids):
        if doc_id in relevant_ids:
            return 1.0 / (i + 1)
    return 0.0


print("Evaluation metrics defined.")


## Benchmark: Raw vs Rewritten Retrieval

Run all four approaches on every query:
1. **Raw** -- no rewriting
2. **Clarification** -- expand pronouns and context
3. **Step-back** -- broaden to general topic
4. **Multi-query** -- split and merge


In [ ]:
methods = {
    "Raw": lambda q: retrieve(q, k=K),
    "Clarify": lambda q: retrieve(rewrite_clarify(q), k=K),
    "Step-back": lambda q: retrieve(rewrite_stepback(q), k=K),
    "Multi-query": lambda q: retrieve_multi(rewrite_multiquery(q), k=K),
}

results = {name: [] for name in methods}

for q in eval_queries:
    query_text = q["query"]
    relevant = q["relevant"]

    for name, search_fn in methods.items():
        hits = search_fn(query_text)
        retrieved_ids = [h["id"] for h in hits]
        results[name].append({
            "query": query_text,
            "type": q["type"],
            "recall": recall_at_k(retrieved_ids, relevant, K),
            "mrr": mrr_score(retrieved_ids, relevant),
            "retrieved": retrieved_ids,
            "relevant": relevant,
        })

print(f"Benchmark complete: {len(eval_queries)} queries x {len(methods)} methods")


## Aggregate Results

In [ ]:
print(f"{'Metric':<12} {'Raw':>8} {'Clarify':>8} {'StepBack':>8} {'MultiQ':>8}")
print("-" * 48)

for metric in ["recall", "mrr"]:
    row = {}
    for name in methods:
        avg = sum(r[metric] for r in results[name]) / len(results[name])
        row[name] = avg
    best_val = max(row.values())
    vals = ""
    for name in methods:
        marker = " *" if row[name] == best_val else ""
        vals += f"{row[name]:>7.3f}{marker}"
    label = f"Recall@{K}" if metric == "recall" else "MRR"
    print(f"{label:<12} {vals}")

# Improvement over raw
print()
raw_recall = sum(r["recall"] for r in results["Raw"]) / len(results["Raw"])
for name in ["Clarify", "Step-back", "Multi-query"]:
    method_recall = sum(r["recall"] for r in results[name]) / len(results[name])
    delta = method_recall - raw_recall
    pct = (delta / raw_recall * 100) if raw_recall > 0 else 0
    direction = "+" if delta >= 0 else ""
    print(f"  {name} vs Raw: {direction}{delta:.3f} ({direction}{pct:.1f}% Recall@{K})")


## Per-Query-Type Breakdown

Which rewriting strategy helps which type of query?


In [ ]:
for qtype in ["vague", "narrow", "multi", "clear"]:
    print(f"\nQuery type: {qtype}")
    print(f"{'Method':<12} {'R@'+str(K):>6} {'MRR':>6}")
    print("-" * 26)
    for name in methods:
        type_results = [r for r in results[name] if r["type"] == qtype]
        if not type_results:
            continue
        avg_r = sum(r["recall"] for r in type_results) / len(type_results)
        avg_m = sum(r["mrr"] for r in type_results) / len(type_results)
        print(f"{name:<12} {avg_r:>6.3f} {avg_m:>6.3f}")


## Per-Query Detail

In [ ]:
print("Per-Query Recall@3:\n")
for i, q in enumerate(eval_queries):
    raw_r = results["Raw"][i]["recall"]
    clarify_r = results["Clarify"][i]["recall"]
    stepback_r = results["Step-back"][i]["recall"]
    multi_r = results["Multi-query"][i]["recall"]
    
    best = max(raw_r, clarify_r, stepback_r, multi_r)
    improved = best > raw_r
    marker = "IMPROVED" if improved else ("same" if best == raw_r else "worse")
    
    print(f"  Q{i+1:2d} [{q['type']:>6s}] Raw={raw_r:.2f}  Clarify={clarify_r:.2f}  "
          f"Step={stepback_r:.2f}  Multi={multi_r:.2f}  [{marker}]")


## Qualitative Comparison: Before and After Rewriting

Let's examine specific queries where rewriting made a difference.


In [ ]:
print("Before/After Comparison")
print("=" * 70)

for i, q in enumerate(eval_queries):
    raw_r = results["Raw"][i]["recall"]
    best_method = max(["Clarify", "Step-back", "Multi-query"],
                      key=lambda m: results[m][i]["recall"])
    best_r = results[best_method][i]["recall"]
    
    if best_r > raw_r:  # improvement
        print(f"\nQ{i+1}: {q['query']}")
        print(f"  Type: {q['type']} | Relevant: {q['relevant']}")
        print(f"  Raw retrieved:  {results['Raw'][i]['retrieved']} (recall={raw_r:.2f})")
        print(f"  {best_method} retrieved: {results[best_method][i]['retrieved']} (recall={best_r:.2f})")
        
        # Show what the rewrite looked like
        if best_method == "Clarify":
            print(f"  Rewritten query: {rewrite_clarify(q['query'])}")
        elif best_method == "Step-back":
            print(f"  Step-back query: {rewrite_stepback(q['query'])}")
        else:
            print(f"  Sub-queries: {rewrite_multiquery(q['query'])}")
        print(f"  Why: {q['note']}")

print("\n" + "=" * 70)


## Error Analysis

Cases where rewriting **hurt** or didn't help.


In [ ]:
print("Error Analysis: Where Rewriting Hurt or Didn't Help\n")

for i, q in enumerate(eval_queries):
    raw_r = results["Raw"][i]["recall"]
    
    for name in ["Clarify", "Step-back", "Multi-query"]:
        method_r = results[name][i]["recall"]
        if method_r < raw_r:  # rewriting hurt
            print(f"  Q{i+1} [{q['type']}]: {name} hurt retrieval")
            print(f"    Raw recall={raw_r:.2f} -> {name} recall={method_r:.2f}")
            print(f"    Query: {q['query']}")
            if name == "Step-back":
                print(f"    Step-back made query too broad, diluting signal")
            elif name == "Clarify":
                print(f"    Clarification added noise or wrong context")
            else:
                print(f"    Sub-queries fragmented the retrieval")
            print()

# Summary: which strategy is safest?
print("\nSafety Analysis (how often does each strategy hurt?):")
for name in ["Clarify", "Step-back", "Multi-query"]:
    hurt_count = sum(1 for i in range(len(eval_queries))
                     if results[name][i]["recall"] < results["Raw"][i]["recall"])
    total = len(eval_queries)
    print(f"  {name}: hurt {hurt_count}/{total} queries ({hurt_count/total:.0%})")


## Recommended Strategy Per Query Type

Based on benchmark results, which strategy works best for each type?


In [ ]:
print("Best Strategy Per Query Type\n")
for qtype in ["vague", "narrow", "multi", "clear"]:
    best_name = None
    best_recall = -1
    for name in methods:
        type_results = [r for r in results[name] if r["type"] == qtype]
        if type_results:
            avg = sum(r["recall"] for r in type_results) / len(type_results)
            if avg > best_recall:
                best_recall = avg
                best_name = name
    print(f"  {qtype:>6s}: {best_name} (Recall@{K} = {best_recall:.3f})")

print("\nRecommendation:")
print("  - Classify incoming query type first")
print("  - Route to the best rewriting strategy")
print("  - Fall back to Raw for already-clear queries")


## Limitations

1. **Rule-based rewriting.** Production systems use an LLM (GPT-4, Claude)
   for context-aware rewriting. Rules are brittle and domain-specific.
2. **No conversation history.** Real chatbots need to resolve "it" using
   the previous turn, not just domain keywords.
3. **Small corpus.** 25 documents are easy; real systems have millions.
   Rewriting impact grows with corpus size.
4. **Single embedding model.** Larger models may reduce the gap between
   raw and rewritten queries.
5. **No reranking.** Cross-encoder reranking after retrieval can compensate
   for imperfect queries without rewriting.
6. **Binary relevance.** Real queries have graded relevance.


## Common Mistakes

| Mistake | Why it's wrong | Fix |
|---------|---------------|-----|
| Rewriting all queries | Clear queries may get worse | Classify first, rewrite only when needed |
| Step-back on everything | Too-broad queries dilute signal | Use only for overly-specific queries |
| Multi-query with k=1 | Sub-queries need k>1 to find all relevant docs | Use k>=3 per sub-query |
| Not evaluating rewriting | Assuming rewriting always helps | Benchmark raw vs rewritten on eval set |
| Ignoring latency cost | LLM rewriting adds 200-500ms | Cache rewrites for repeated queries |
| Domain-blind rewriting | Generic rewrites miss domain context | Inject domain knowledge into rewrite prompts |


## Mini Challenge

1. **LLM-powered rewriting.** Replace rule-based `rewrite_clarify()` with
   a real LLM call. Compare quality vs rule-based.

2. **Conversation-aware rewriting.** Add a `history` parameter that carries
   the previous user/assistant turns. Use history to resolve pronouns.

3. **Adaptive routing.** Build a classifier that detects query type
   (vague/narrow/multi/clear) and routes to the best strategy automatically.

4. **Combine with hybrid search.** Add BM25 + RRF from project 21.
   Does rewriting + hybrid beat either alone?

5. **HyDE (Hypothetical Document Embedding).** Generate a hypothetical
   answer, embed it, and use that embedding for retrieval.


## Production Considerations

| Aspect | Approach |
|--------|----------|
| **Rewriting latency** | LLM rewrite adds 200-500ms; cache common rewrites |
| **Query classification** | Lightweight classifier to decide if rewriting is needed |
| **A/B testing** | Compare raw vs rewritten in production with user satisfaction metrics |
| **Cost** | One extra LLM call per query; use smaller/faster model for rewriting |
| **Monitoring** | Log original + rewritten queries to track rewrite quality |
| **Feedback loop** | If users report bad answers, check if rewriting was the cause |
| **Fallback** | If rewrite fails or times out, fall back to raw query |


## How to Improve This Project

1. **Use an LLM for rewriting** -- GPT-4 or Claude can handle pronoun
   resolution, context inference, and query decomposition much better.
2. **Add conversation history** -- Real chatbots need multi-turn context.
3. **Implement HyDE** -- Generate a hypothetical answer, embed it,
   and use that for retrieval.
4. **Combine strategies** -- Run clarification + multi-query together
   and merge all results.
5. **Scale corpus** -- Test on 1000+ documents to see how rewriting
   impact changes.
6. **Add cross-encoder reranking** -- Rerank after retrieval to further
   improve quality.


## Key Takeaways

1. **Users write bad search queries.** Vague pronouns, missing context,
   and multi-part questions are the norm, not the exception.

2. **Query rewriting dramatically improves retrieval.** Even simple
   rule-based rewriting can boost recall by 15-50%.

3. **Different query problems need different strategies.**
   Clarification for vague queries, step-back for narrow queries,
   multi-query for complex queries.

4. **Rewriting can hurt clear queries.** Don't rewrite everything.
   Classify first, then decide.

5. **Rewriting is cheap insurance.** One extra LLM call is far cheaper
   than a bad answer that requires human correction.

6. **Composable with other RAG techniques.** Rewriting + hybrid search
   + reranking + compression = production-grade RAG pipeline.

7. **Always benchmark.** Never assume rewriting helps -- measure it
   on your actual queries and corpus.
